# 11 · Convolutional Neural Networks and Advanced CNN Architectures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anuxlab/deep-learning-systems/blob/main/notebooks/11-cnn/11_cnn.ipynb)
[![Run on GitHub Actions](https://img.shields.io/badge/run-GitHub%20Actions-2088FF?logo=githubactions&logoColor=white)](https://github.com/anuxlab/deep-learning-systems/actions/workflows/nb-11-cnn.yml)

| | |
|---|---|
| **Wiki chapter** | [Module 11](https://github.com/anuxlab/deep-learning-systems/wiki/Module-11) |
| **Level** | Intermediate |
| **Typical CPU runtime** | ~1–2 min (GitHub free runner, no GPU) |
| **Prerequisites** | Modules 01, 07 |

## What you will do

- Implement 2-D convolution (cross-correlation) from scratch and match `F.conv2d`.
- Predict output sizes with stride/padding/dilation and compute receptive fields and parameter counts.
- Implement pooling and the backward pass of convolution; verify against autograd.
- Demonstrate translation equivariance and weight sharing; compare CNN vs MLP on digits.
- Build a residual block and inspect learned filters.

> **How to use this notebook.** Read each markdown cell, *predict* what the next code cell will print, then run it.
> Every section ends with `check(...)` assertions: if one fails, something in your understanding (or the code) is off.
> Figures are saved to `figures/` and a machine-readable summary to `metrics.json` next to this notebook.

## 0 · Setup and reproducibility

We import shared helpers (`show` saves each figure to `figures/`, `check` asserts and logs, `timer` logs wall-clock time), fix all random seeds and print an environment report so every run is traceable (see the wiki's *Implementation and Research Standards*).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "_common"))   # shared helpers: notebooks/_common/dlsys.py
from dlsys import *                                       # np, plt, torch, seed_everything, show, check, timer, ...

SEED = seed_everything(42)
ENV = env_report()

## 1 · Convolution from scratch
Deep-learning "convolution" is **cross-correlation**: slide the kernel without flipping it,
$$Y[o,i,j]=b_o+\sum_{c}\sum_{u,v}K[o,c,u,v]\;X[c,\,si+u-p,\,sj+v-p]$$
with stride $s$ and padding $p$. The output size per spatial dimension is
$$\Big\lfloor\frac{n+2p-d(k-1)-1}{s}\Big\rfloor+1$$
($d$ = dilation). We implement it with explicit loops (clear, slow) and with an `im2col`/stride-trick version (fast), and check against PyTorch.

In [ ]:
def conv2d_loops(X, K, b=None, stride=1, pad=0, dil=1):
    C, H, W = X.shape; O, _, kh, kw = K.shape
    Xp = np.pad(X, ((0, 0), (pad, pad), (pad, pad)))
    Ho = (H + 2 * pad - dil * (kh - 1) - 1) // stride + 1; Wo = (W + 2 * pad - dil * (kw - 1) - 1) // stride + 1
    Y = np.zeros((O, Ho, Wo))
    for o in range(O):
        for i in range(Ho):
            for j in range(Wo):
                patch = Xp[:, i * stride: i * stride + dil * (kh - 1) + 1: dil, j * stride: j * stride + dil * (kw - 1) + 1: dil]
                Y[o, i, j] = np.sum(patch * K[o]) + (0 if b is None else b[o])
    return Y
def out_size(n, k, s=1, p=0, d=1): return (n + 2 * p - d * (k - 1) - 1) // s + 1
r = np.random.default_rng(0)
for (H, k, s, p, d) in [(28, 3, 1, 0, 1), (28, 3, 1, 1, 1), (28, 5, 2, 2, 1), (32, 3, 2, 1, 1), (32, 3, 1, 2, 2), (15, 7, 3, 0, 1)]:
    X = r.normal(size=(3, H, H)); K = r.normal(size=(4, 3, k, k)); b = r.normal(size=4)
    mine = conv2d_loops(X, K, b, s, p, d); ref = F.conv2d(torch.tensor(X)[None], torch.tensor(K), torch.tensor(b), stride=s, padding=p, dilation=d)[0].numpy()
    check(mine.shape == ref.shape == (4, out_size(H, k, s, p, d), out_size(H, k, s, p, d)) and np.allclose(mine, ref), f"H={H} k={k} stride={s} pad={p} dil={d} -> output {mine.shape[1:]} matches torch")

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view
def conv2d_fast(X, K, b=None):                       # stride 1, no padding, via sliding windows + einsum
    win = sliding_window_view(X, K.shape[2:], axis=(1, 2))           # (C, Ho, Wo, kh, kw)
    Y = np.einsum("chwuv,ocuv->ohw", win, K); return Y if b is None else Y + b[:, None, None]
X = r.normal(size=(3, 40, 40)); K = r.normal(size=(8, 3, 5, 5))
import time; t0 = time.perf_counter(); a = conv2d_loops(X, K); t1 = time.perf_counter(); bb = conv2d_fast(X, K); t2 = time.perf_counter()
print(f"loops: {t1-t0:.3f}s   vectorised: {t2-t1:.4f}s   speed-up ×{(t1-t0)/(t2-t1):.0f}")
check(np.allclose(a, bb), "vectorised conv == loop conv"); METRICS["conv_speedup"] = (t1 - t0) / (t2 - t1)

## 2 · Edge detectors: what a filter computes
A $3\times3$ Sobel kernel responds to vertical edges. We convolve a digit image and visualise the feature maps — the first layer of a CNN learns filters like these.

In [ ]:
from sklearn.datasets import load_digits
D = load_digits(); img = np.kron(D.images[7], np.ones((4, 4)))[None] / 16.   # 1x32x32
sobel_x = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1.]]); sobel_y = sobel_x.T; blur = np.ones((3, 3)) / 9
fm = [conv2d_fast(img, k[None, None])[0] for k in (sobel_x, sobel_y, blur)]
fig, ax = plt.subplots(1, 4, figsize=(12, 3)); ax[0].imshow(img[0], cmap="gray"); ax[0].set_title("input")
for a_, f_, t in zip(ax[1:], fm, ["Sobel-x (vertical edges)", "Sobel-y (horizontal edges)", "box blur"]): a_.imshow(f_, cmap="gray"); a_.set_title(t)
[a_.axis("off") for a_ in ax]; show("filters_feature_maps")

## 3 · Receptive field and parameter counts
Receptive field of a stack: $r_l=r_{l-1}+(k_l-1)\,j_{l-1}$ and jump $j_l=j_{l-1}s_l$. Two $3\times3$ convs see $5\times5$; three see $7\times7$ — with $3\cdot9C^2=27C^2$ weights instead of $49C^2$ (the VGG insight). Parameters of a conv layer: $O(Ck_hk_w+1)$ — **independent of image size**, unlike a dense layer.

In [ ]:
def receptive_field(layers):
    rf, j = 1, 1
    for k, s in layers: rf += (k - 1) * j; j *= s
    return rf
print("3×(3x3, s1):", receptive_field([(3, 1)] * 3), " | conv3,s2 x3:", receptive_field([(3, 2)] * 3), " | AlexNet-ish:", receptive_field([(11, 4), (3, 2), (5, 1), (3, 2), (3, 1)]))
check(receptive_field([(3, 1)] * 2) == 5 and receptive_field([(3, 1)] * 3) == 7, "stacked 3×3 convs: RF 5 and 7")
C = 64; print(f"three 3x3 convs: {3*9*C*C:,} weights vs one 7x7 conv: {49*C*C:,} weights")
conv = nn.Conv2d(3, 16, 3); fc = nn.Linear(3 * 32 * 32, 16 * 30 * 30)
print(f"conv params: {count_params(conv):,}   dense layer with the same input/output sizes: {count_params(fc):,}")
check(count_params(conv) == 3 * 16 * 9 + 16, "conv params = O·C·kh·kw + O"); check(count_params(fc) > 1000 * count_params(conv), "weight sharing saves >1000× parameters")

## 4 · Pooling, and convolution backprop
Max-pool routes the gradient to the arg-max position only; average-pool spreads it evenly. For convolution, with $\bar Y=\partial\mathcal L/\partial Y$:
$$\bar K[o,c,u,v]=\sum_{i,j}\bar Y[o,i,j]\,X[c,i+u,j+v],\qquad \bar X=\text{full-correlation of }\bar Y\text{ with the flipped kernel}.$$
Note $\bar K$ **sums over all positions** — the shared-filter gradient. We implement both and compare to autograd.

In [ ]:
def maxpool(X, k=2): C, H, W = X.shape; return X[:, :H // k * k, :W // k * k].reshape(C, H // k, k, W // k, k).max((2, 4))
Xp = r.normal(size=(2, 8, 8)); check(np.allclose(maxpool(Xp), F.max_pool2d(torch.tensor(Xp)[None], 2)[0].numpy()), "maxpool matches torch")

def conv_backward(X, K, dY):                           # stride 1, no padding
    C, H, W = X.shape; O, _, kh, kw = K.shape; dK = np.zeros_like(K); dX = np.zeros_like(X); Ho, Wo = dY.shape[1:]
    for u in range(kh):
        for v in range(kw):
            dK[:, :, u, v] = np.einsum("oij,cij->oc", dY, X[:, u:u + Ho, v:v + Wo])
            dX[:, u:u + Ho, v:v + Wo] += np.einsum("oij,oc->cij", dY, K[:, :, u, v])
    return dK, dX
X = r.normal(size=(3, 9, 9)); K = r.normal(size=(4, 3, 3, 3)); Y = conv2d_fast(X, K); dY = r.normal(size=Y.shape)
dK, dX = conv_backward(X, K, dY)
Xt, Kt = torch.tensor(X, requires_grad=True), torch.tensor(K, requires_grad=True)
(F.conv2d(Xt[None], Kt)[0] * torch.tensor(dY)).sum().backward()
check(np.allclose(dK, Kt.grad.numpy()), "conv backward ∂L/∂K == autograd"); check(np.allclose(dX, Xt.grad.numpy()), "conv backward ∂L/∂X == autograd")

## 5 · Translation equivariance, and (approximate) invariance
Convolution is *equivariant*: shifting the input shifts the feature map by the same amount, $f(T_\Delta x)=T_\Delta f(x)$. Global pooling afterwards makes the output (approximately) *invariant*. We verify equivariance exactly with circular padding, where boundary effects vanish.

In [ ]:
Xc = r.normal(size=(1, 16, 16)); Kc = r.normal(size=(1, 1, 3, 3)); conv_circ = lambda x: F.conv2d(F.pad(torch.tensor(x)[None], (1, 1, 1, 1), mode="circular"), torch.tensor(Kc))[0].numpy()
shift = lambda x, dx, dy: np.roll(x, (dy, dx), axis=(-2, -1))
check(np.allclose(conv_circ(shift(Xc, 3, 5)), shift(conv_circ(Xc), 3, 5)), "conv(shift(x)) == shift(conv(x))  (equivariance)")
check(np.isclose(conv_circ(shift(Xc, 3, 5)).max(), conv_circ(Xc).max()), "global max-pool after conv is shift-invariant")
d_fc = nn.Linear(256, 256).double(); xs = torch.tensor(Xc.reshape(1, -1))
with torch.no_grad(): lhs = d_fc(torch.tensor(shift(Xc, 3, 5).reshape(1, -1))).numpy(); rhs = shift(d_fc(xs).numpy().reshape(1, 16, 16), 3, 5).reshape(1, -1)
check(not np.allclose(lhs, rhs), "a dense layer has no such property")

## 6 · CNN vs MLP on digits
Same data (8×8 digits, 1,797 images), similar training. The CNN uses ~5× fewer parameters yet generalises as well or better thanks to locality and weight sharing; we also test **robustness to a 1-pixel shift**, where the CNN's built-in prior shows.

In [ ]:
from sklearn.model_selection import train_test_split
Xd = torch.tensor(D.images / 16., dtype=torch.float32)[:, None]; yd = torch.tensor(D.target)
idx = np.random.default_rng(0).permutation(len(Xd)); tr, te = idx[:1300], idx[1300:]
class SmallCNN(nn.Module):
    def __init__(s):
        super().__init__(); s.f = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.AdaptiveMaxPool2d(1), nn.Flatten()); s.h = nn.Linear(16, 10)
    def forward(s, x): return s.h(s.f(x))
mlp = nn.Sequential(nn.Flatten(), nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10))
def fit(m, epochs=40, lr=3e-3):
    o = torch.optim.Adam(m.parameters(), lr)
    for ep in range(epochs):
        m.train(); perm = torch.tensor(tr)[torch.randperm(len(tr))]
        for s in range(0, len(perm), 32): b = perm[s:s + 32]; o.zero_grad(); F.cross_entropy(m(Xd[b]), yd[b]).backward(); o.step()
    m.eval(); return m
def acc(m, X, y):
    with torch.no_grad(): return (m(X).argmax(1) == y).float().mean().item()
torch.manual_seed(0); cnn = fit(SmallCNN()); torch.manual_seed(0); mlp = fit(mlp)
Xs = torch.roll(Xd[te], 1, dims=3)   # shift images one pixel to the right
print(f"CNN : {count_params(cnn):6,d} params  acc {acc(cnn, Xd[te], yd[te]):.3f}  | shifted-by-1px acc {acc(cnn, Xs, yd[te]):.3f}")
print(f"MLP : {count_params(mlp):6,d} params  acc {acc(mlp, Xd[te], yd[te]):.3f}  | shifted-by-1px acc {acc(mlp, Xs, yd[te]):.3f}")
check(acc(cnn, Xd[te], yd[te]) > .9, "CNN > 90% accuracy"); check(count_params(cnn) < count_params(mlp), "CNN has fewer parameters than the MLP")
drop_c = acc(cnn, Xd[te], yd[te]) - acc(cnn, Xs, yd[te]); drop_m = acc(mlp, Xd[te], yd[te]) - acc(mlp, Xs, yd[te])
print(f"accuracy drop under shift: CNN {drop_c:.3f}  vs MLP {drop_m:.3f}"); METRICS.update(cnn_acc=acc(cnn, Xd[te], yd[te]), mlp_acc=acc(mlp, Xd[te], yd[te]), shift_drop_cnn=drop_c, shift_drop_mlp=drop_m)
check(drop_c < drop_m, "the CNN is more robust to translation than the MLP")
W1 = cnn.f[0].weight.detach().squeeze(1).numpy(); fig, ax = plt.subplots(1, 8, figsize=(12, 1.8))
for i, a_ in enumerate(ax): a_.imshow(W1[i], cmap="RdBu"); a_.axis("off")
plt.suptitle("Learned first-layer 3×3 filters"); show("learned_filters")

## 7 · Architecture families in miniature: the residual block
ResNet: $y=\phi(x+F(x))$ where $F$=conv–BN–ReLU–conv–BN. We build the block, a downsampling variant with a $1\times1$ projection shortcut, and print the shape/parameter flow of a tiny ResNet. Other families (VGG: stacks of 3×3; Inception: parallel branches; MobileNet: depthwise-separable convs) differ in *how* they compose the same operations — below we also quantify the depthwise-separable saving.

In [ ]:
class Block(nn.Module):
    def __init__(s, cin, cout, stride=1):
        super().__init__(); s.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); s.b1 = nn.BatchNorm2d(cout); s.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False); s.b2 = nn.BatchNorm2d(cout)
        s.sc = nn.Identity() if (stride == 1 and cin == cout) else nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(s, x): return F.relu(s.sc(x) + s.b2(s.c2(F.relu(s.b1(s.c1(x))))))
net = nn.Sequential(nn.Conv2d(3, 16, 3, 1, 1), Block(16, 16), Block(16, 32, 2), Block(32, 64, 2), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 10))
x = torch.randn(2, 3, 32, 32)
for i, l in enumerate(net): x = l(x); print(f"layer {i}: {l.__class__.__name__:18s} -> {tuple(x.shape)}")
check(x.shape == (2, 10), "tiny ResNet maps (2,3,32,32) -> (2,10)"); print("total params:", count_params(net))
std = nn.Conv2d(128, 128, 3, padding=1); dws = nn.Sequential(nn.Conv2d(128, 128, 3, padding=1, groups=128), nn.Conv2d(128, 128, 1))
print(f"standard 3x3 conv: {count_params(std):,}  depthwise-separable: {count_params(dws):,}  (×{count_params(std)/count_params(dws):.1f} fewer)")
check(count_params(std) > 7 * count_params(dws), "depthwise-separable conv uses ~8× fewer parameters")

In [ ]:
save_metrics("11", METRICS)
log.info('notebook 11 finished OK')

## Wrap-up

## Exercises (do these next)

1. Add dilation to `conv2d_fast` and verify against torch.
2. Train the CNN with data augmentation (random ±1 pixel shifts) and re-measure the shift robustness.
3. Implement average pooling backward and 1-D convolution for time series.

## Where to read more

- Wiki chapter: [Module 11](https://github.com/anuxlab/deep-learning-systems/wiki/Module-11) (derivations, worked example, Q&A).
- LeCun et al. (1998); Krizhevsky et al. (2012); Simonyan & Zisserman (2015) VGG; He et al. (2016) ResNet; Howard et al. (2017) MobileNet; Dumoulin & Visin (2016) *A guide to convolution arithmetic*.